In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7efd73d5-a537-4167-ab3a-d0f62f19b174;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 679ms :: artifacts dl 47ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
df_geolocation = spark.read.option('delimiter', ',') \
              .option('header', 'true') \
              .option('nullValue', 'NULL') \
              .csv('s3a://last-mile-optimization-raw/dataset-orders/olist_geolocation_dataset.csv')
df_geolocation.show()

26/04/04 22:24:11 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------------------------+-------------------+-------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+-------------------+----------------+-----------------+
|                      01037| -23.54562128115268| -46.63929204800168|       sao paulo|               SP|
|                      01046|-23.546081127035535| -46.64482029837157|       sao paulo|               SP|
|                      01046| -23.54612896641469| -46.64295148361138|       sao paulo|               SP|
|                      01041|  -23.5443921648681| -46.63949930627844|       sao paulo|               SP|
|                      01035|-23.541577961711493| -46.64160722329613|       sao paulo|               SP|
|                      01012|-23.547762303364266| -46.63536053788448|       são paulo|               SP|
|                      01047|-23.546273112412678| -46.6

In [4]:
from pyspark.sql.functions import col, upper

# Filtrar por estado 'SP'
df_sp = df_geolocation.filter(col("geolocation_state") == "SP")

# Colocar nomes de cidade em letra maiúscula 
df_geolocation = df_sp.withColumn('geolocation_city', upper(col('geolocation_city')))

# Remover a coluna customer_state
df_geolocation = df_geolocation.drop("geolocation_city")
df_geolocation = df_geolocation.drop("geolocation_state")

df_geolocation.show()

+---------------------------+-------------------+-------------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|
+---------------------------+-------------------+-------------------+
|                      01037| -23.54562128115268| -46.63929204800168|
|                      01046|-23.546081127035535| -46.64482029837157|
|                      01046| -23.54612896641469| -46.64295148361138|
|                      01041|  -23.5443921648681| -46.63949930627844|
|                      01035|-23.541577961711493| -46.64160722329613|
|                      01012|-23.547762303364266| -46.63536053788448|
|                      01047|-23.546273112412678| -46.64122516971552|
|                      01013|-23.546923208436723|  -46.6342636964915|
|                      01029|-23.543769055769133| -46.63427784085132|
|                      01011|-23.547639550320632| -46.63603162315495|
|                      01013|-23.547325128224376| -46.63418378613892|
|                   

In [5]:
df_geolocation.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/geolocation_cleaned_dataset.csv')

spark.stop()

26/04/04 22:24:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/04 22:24:23 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
